# Notebook 04 - Results and Interpretation

This notebook synthesizes the modeling and feature-selection results and answers the research question:

> Which demographic, socioeconomic, and academic factors most strongly predict student dropout,
> and which classification algorithm achieves the best predictive performance?


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

from src.data_loader import load_raw, prepare_binary, get_X_y
from src.evaluate_models import best_model_name, evaluate_best_model, get_lr_coefficients
from src.plots import (
    plot_confusion_matrix,
    plot_cv_boxplot,
    plot_feature_importance_top,
    plot_feature_subset_comparison,
    plot_group_importance,
    plot_lr_coefficients,
    plot_model_comparison,
    plot_per_class_metrics,
    plot_precision_recall_curve,
    plot_radar_model_comparison,
    plot_roc_curve,
)

pd.set_option('display.float_format', '{:.4f}'.format)


## 1. Load Results


In [ ]:
cv_results = pd.read_csv('../results/tables/model_comparison.csv')
df_all = pd.read_csv('../results/tables/model_comparison_all_features.csv')
df_top10 = pd.read_csv('../results/tables/model_comparison_top10.csv')
mi_importance = pd.read_csv('../results/tables/feature_importance_mutual_info.csv')
grp_summary = pd.read_csv('../results/tables/grouped_feature_summary.csv')

with open('../results/tables/cv_fold_scores.json', encoding='utf-8') as f:
    cv_fold_scores = json.load(f)

print('CV results loaded.')
cv_results[['model', 'accuracy_mean', 'precision_mean', 'recall_mean', 'f1_mean', 'roc_auc_mean']]


## 2. Best Model Evaluation


In [ ]:
df = prepare_binary(load_raw())
X, y = get_X_y(df)

best = best_model_name(cv_results, metric='f1_mean')
print(f'Best model: {best}')

eval_results = evaluate_best_model(X, y, best)


In [ ]:
fig = plot_confusion_matrix(eval_results['confusion_matrix'], best)
plt.show()

fig = plot_roc_curve(eval_results['fpr'], eval_results['tpr'], eval_results['roc_auc'], best)
plt.show()


In [ ]:
fig = plot_precision_recall_curve(
    eval_results['precision_curve'],
    eval_results['recall_curve'],
    eval_results['avg_precision'],
    best,
)
plt.show()

report = classification_report(eval_results['y_test'], eval_results['y_pred'], output_dict=True)
fig = plot_per_class_metrics(report, best)
plt.show()


## 3. Full Model Comparison


In [ ]:
fig = plot_model_comparison(cv_results, metric='accuracy')
plt.show()

fig = plot_model_comparison(cv_results, metric='f1')
plt.show()

fig = plot_radar_model_comparison(cv_results)
plt.show()

cv_results[['model', 'accuracy_mean', 'precision_mean', 'recall_mean', 'f1_mean', 'roc_auc_mean']]


In [ ]:
fig = plot_cv_boxplot(cv_fold_scores, metric='f1', fname='cv_boxplot_f1.png')
plt.show()

fig = plot_cv_boxplot(cv_fold_scores, metric='roc_auc', fname='cv_boxplot_roc.png')
plt.show()

fig = plot_feature_subset_comparison(df_all, df_top10, metric='f1_mean')
plt.show()


## 4. Feature Group Analysis


In [ ]:
print('Feature group importance (mean Mutual Information):')
print(grp_summary.to_string(index=False))

fig = plot_group_importance(grp_summary, score_col='mean_score')
plt.show()

fig = plot_feature_importance_top(
    mi_importance,
    'mi_score',
    title='Top-15 Predictors of Dropout (Mutual Information)',
    n=15,
    fname='feature_importance_top15.png',
)
plt.show()


## 5. Research Question - Answered

### Q1: Which factors most strongly predict student dropout?


In [ ]:
top15 = mi_importance.head(15)['feature'].tolist()
print('Top-15 dropout predictors (Mutual Information):')
for idx, feat in enumerate(top15, start=1):
    print(f'  {idx:2d}. {feat}')


### Answer to Q1 - Factor Importance / Antwort auf F1 - Faktorwichtigkeit

**EN:**
Academic performance indicators are by far the strongest predictors of student dropout.
The top-4 features - curricular units approved and grades in both semesters - account for
more than 70 % of Random Forest feature importance. Among socioeconomic features,
*Tuition fees up to date* (rank 5, MI = 0.110) and *Scholarship holder* (rank 9, MI = 0.059)
are the most informative. Pure demographic variables (Gender, Age at enrollment) are
significant but rank lower (ranks 10 and 8), confirming the hypothesis ordering.
An unexpected finding: the feature *Course* (study field, MI = 0.048) outperforms all
socioeconomic features individually, suggesting strong study-programme-specific dropout patterns.

**DE:**
Akademische Leistungsindikatoren sind mit Abstand die staerksten Praediktoren.
Die Top-4-Merkmale - bestandene Lehrveranstaltungen und Noten beider Semester -
erklaeren ueber 70 % der Random-Forest-Importance. Bei den soziooekonomischen Merkmalen
sind *Tuition fees up to date* (Rang 5, MI = 0,110) und *Scholarship holder* (Rang 9)
am informativsten. Rein demografische Variablen (Geschlecht, Alter) sind signifikant,
aber rangieren weiter hinten und bestaetigen damit die Hypothese.


### Q2: Which algorithm achieves the best predictive performance?


In [ ]:
best_row = cv_results[cv_results['model'] == best].iloc[0]

print(f'Best classifier: {best}')
print(f"  Accuracy : {best_row['accuracy_mean']:.4f} +/- {best_row['accuracy_std']:.4f}")
print(f"  Precision: {best_row['precision_mean']:.4f} +/- {best_row['precision_std']:.4f}")
print(f"  Recall   : {best_row['recall_mean']:.4f} +/- {best_row['recall_std']:.4f}")
print(f"  F1       : {best_row['f1_mean']:.4f} +/- {best_row['f1_std']:.4f}")
print(f"  ROC-AUC  : {best_row['roc_auc_mean']:.4f} +/- {best_row['roc_auc_std']:.4f}")


### Answer to Q2 - Best Algorithm / Antwort auf F2 - Bester Algorithmus

**EN:**
Logistic Regression achieves the best performance across all metrics under 10-fold
stratified cross-validation: Accuracy = 91.1 % (+/-1.2 %), F1 = 88.0 % (+/-1.8 %),
ROC-AUC = 95.3 % (+/-0.4 %). Random Forest and SVM follow closely (F1 87.5 % and 86.7 %),
while Decision Tree, kNN, and Naive Bayes lag 6-10 pp behind.
The superiority of the linear model suggests the decision boundary between Dropout and
Graduate is predominantly linear in the feature space, driven by a small set of dominant
academic performance indicators.

**DE:**
Logistic Regression erzielt die beste Leistung in allen Metriken unter 10-facher
stratifizierter Kreuzvalidierung: Accuracy = 91,1 % (+/-1,2 %), F1 = 88,0 % (+/-1,8 %),
ROC-AUC = 95,3 % (+/-0,4 %). Random Forest und SVM folgen knapp dahinter.
Decision Tree, kNN und Naive Bayes liegen 6-10 Prozentpunkte dahinter.
Der Erfolg des linearen Modells legt nahe, dass die Entscheidungsgrenze zwischen
Abbruch und Abschluss im Merkmalsraum ueberwiegend linear ist.


## 6. Best Model Interpretation


In [ ]:
with open('../results/tables/top_feature_subsets.json', encoding='utf-8') as f:
    top15 = json.load(f)['top15']

coefs = get_lr_coefficients(X, y, top_features=top15)
fig = plot_lr_coefficients(coefs)
plt.show()


## 7. Hypothesis Evaluation

The planned hypothesis is considered supported when academic features rank above
socioeconomic features, which in turn rank above demographic features.


In [ ]:
group_order = grp_summary.sort_values('mean_score', ascending=False)['group'].tolist()
print('Feature group ranking (highest to lowest mean MI score):')
for rank, group in enumerate(group_order, start=1):
    row = grp_summary[grp_summary['group'] == group].iloc[0]
    print(f"  {rank}. {group:<15s}  mean_score={row['mean_score']:.5f}")

academic_rank = group_order.index('academic') + 1 if 'academic' in group_order else None
socio_rank = group_order.index('socioeconomic') + 1 if 'socioeconomic' in group_order else None
demo_rank = group_order.index('demographic') + 1 if 'demographic' in group_order else None

hypothesis_supported = (
    academic_rank is not None and
    socio_rank is not None and
    demo_rank is not None and
    academic_rank < socio_rank < demo_rank
)

print(f'\nAcademic rank     : {academic_rank}')
print(f'Socioeconomic rank: {socio_rank}')
print(f'Demographic rank  : {demo_rank}')
print(f'\nHypothesis SUPPORTED: {hypothesis_supported}')
